In [ ]:
import tensorflow as tf

import pandas as pd
import numpy as np

import os

In [ ]:
DATA_DIR = '../input/feedback-prize-english-language-learning'

In [ ]:
train_data = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_data = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

sample_submission = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print(f'Training data shape: {train_data.shape}')
print(f'Testing data shape:: {test_data.shape}')

In [ ]:
target_columns = train_data.drop(['text_id', 'full_text'], axis=1).columns

In [ ]:
def load_data(df):
    return list(df['full_text'].values), list(df[target_columns].values)

def train_test_split(data, validation_split=0.1):
    text, targets = load_data(data)
    n_val = int(len(text) * validation_split)
    return text[:-n_val], targets[:-n_val], text[-n_val:], targets[-n_val:]

In [ ]:
validation_split = 0.1
train_text, train_targets, val_text, val_targets = train_test_split(train_data, validation_split)

In [ ]:
# Create training and validation tf datasets.
BUFFER_SIZE_TRAIN = len(train_text)
BUFFER_SIZE_VAL = len(val_text)
BATCH_SIZE = 64

train_dataset = tf.data.Dataset.from_tensor_slices((train_text, train_targets)).shuffle(BUFFER_SIZE_TRAIN)
train_dataset = train_dataset.batch(BATCH_SIZE)

validation_dataset = tf.data.Dataset.from_tensor_slices((val_text, val_targets)).shuffle(BUFFER_SIZE_VAL)
validation_dataset = validation_dataset.batch(BATCH_SIZE)

In [ ]:
max_tokens = 1024
output_sequence_length = int(train_data['full_text'].map(len).max())

text_vectorizer = tf.keras.layers.TextVectorization(max_tokens=max_tokens,
                                                   standardize='lower_and_strip_punctuation',
                                                   split='whitespace',
                                                    output_sequence_length=output_sequence_length
                                                   )

text_vectorizer.adapt(train_text)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Transform string to tokens.
train_ds = train_dataset.map(lambda text, targets: (text_vectorizer(text), targets / 5))
val_ds = validation_dataset.map(lambda text, targets: (text_vectorizer(text), targets / 5))


# Optimize the datasets for performance.
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
def load_test(df):
    text = list(df['full_text'].values)
    dataset = tf.data.Dataset.from_tensor_slices((text,))
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.map(lambda text: text_vectorizer(text)).cache().prefetch(AUTOTUNE)
    return dataset

test_ds = load_test(test_data)

In [ ]:
embedding_dim = 16

def get_model():
    model = tf.keras.Sequential([tf.keras.layers.Input((output_sequence_length,)),
                                tf.keras.layers.Embedding(text_vectorizer.vocabulary_size(),
                                                         embedding_dim),
                                tf.keras.layers.GlobalAveragePooling1D(),
                                tf.keras.layers.Dense(len(target_columns))])
    
    model.compile(optimizer='adam',
                  loss=tf.keras.losses.MeanSquaredError(),
                  metrics=[tf.keras.metrics.RootMeanSquaredError()])
    return model

In [ ]:
model = get_model()
model.summary()

In [ ]:
epochs = 500
es_callback = tf.keras.callbacks.EarlyStopping(patience=5)
validation_split = 0.1

model.fit(train_ds, epochs=epochs, validation_data=val_ds, verbose=2, callbacks=[es_callback])

In [ ]:
prediction = model.predict(test_ds) * 5

In [ ]:
submission = sample_submission.copy()
submission[target_columns] = prediction
submission.to_csv('submission.csv', index=False)